# Phase 5 — Calculs détaillés

Ce notebook reprend l'arithmétique des trois portefeuilles à partir des cours CHF, du SARON et des poids publiés.

Il ne choisit pas de nouveaux poids. Les poids viennent de `data/04_portfolios/portfolio_weights.csv`, produits par :

```text
python -m src.portfolio_construction.build_portfolios
```

La méthode et les formules sont dans `documentation/05_portfolio_construction.ipynb`. La lecture des écarts est dans `notebooks/05_analyse_comparative.ipynb`.

L'exemple suit un titre présent dans les trois portefeuilles, sur le premier mois de la fenêtre, puis les mêmes formules sur tous les mois de la fenêtre. Si NOVN.SW est retenu, l'exemple le suit. Sinon il suit le premier titre retenu.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import MARKET_HISTORY_YEARS, REPORTING_CURRENCY

from src.preparation.quality_treatment import MARKET_PRICES_CHF, SARON_TREATED
from src.portfolio_construction.build_portfolios import (
    MEMBERSHIP_FILE,
    METRICS_FILE,
    MONTHLY_RETURNS_FILE,
    WEIGHTS_FILE,
)

prices = pd.read_csv(MARKET_PRICES_CHF, usecols=["symbol", "timestamp", "close_chf"])
saron = pd.read_csv(SARON_TREATED)
membership = pd.read_csv(MEMBERSHIP_FILE, dtype=str, keep_default_na=False)
weights = pd.read_csv(WEIGHTS_FILE)
monthly = pd.read_csv(MONTHLY_RETURNS_FILE)
metrics = pd.read_csv(METRICS_FILE)

included = membership.loc[membership["decision"].eq("included"), "yahoo_symbol"].tolist()
symbol = "NOVN.SW" if "NOVN.SW" in included else included[0]
print(f"Titres retenus : {len(included)}")
print(f"Titre suivi : {symbol}")
print(f"Mois publiés : {monthly['month'].min()} à {monthly['month'].max()}")


Titres retenus : 83
Titre suivi : NOVN.SW
Mois publiés : 2016-10 à 2026-09


## 1. Fenêtre

Le dernier mois du SARON fixe la fin. Les rendements couvrent les dix années qui s'y terminent. Le cours du mois précédent est nécessaire pour le premier rendement.

In [2]:
end = pd.Period(saron["date"].max(), freq="M")
start = end - (MARKET_HISTORY_YEARS * 12 - 1)
return_months = pd.period_range(start, end, freq="M")
price_months = pd.period_range(start - 1, end, freq="M")
first_return = str(return_months[0])
print(f"Dernier SARON : {end}")
print(f"Mois de rendement : {return_months[0]} à {return_months[-1]} ({len(return_months)} mois)")
print(f"Cours de fin de mois exigés : {price_months[0]} à {price_months[-1]} ({len(price_months)} mois)")


Dernier SARON : 2026-09
Mois de rendement : 2016-10 à 2026-09 (120 mois)
Cours de fin de mois exigés : 2016-09 à 2026-09 (121 mois)


## 2. Cours de fin de mois et rendement d'un titre

Pour chaque titre, un seul cours est gardé par mois : le dernier `close_chf` du mois. Le rendement du premier mois est le cours de fin de ce mois divisé par le cours de fin du mois précédent, moins un.

$$
R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1
$$

In [3]:
prices["timestamp"] = pd.to_datetime(prices["timestamp"])
prices["month"] = prices["timestamp"].dt.to_period("M")
levels = (
    prices.sort_values(["symbol", "timestamp"])
    .groupby(["symbol", "month"], as_index=False)
    .tail(1)
)
wide = levels.pivot(index="month", columns="symbol", values="close_chf")
wide = wide.reindex(index=price_months, columns=included)
asset_returns = wide.pct_change().iloc[1:].copy()
asset_returns.index = asset_returns.index.astype(str)

previous_price = float(wide.loc[price_months[0], symbol])
current_price = float(wide.loc[price_months[1], symbol])
manual_return = current_price / previous_price - 1
panel_return = float(asset_returns.loc[first_return, symbol])

print(f"{symbol}")
print(f"Cours fin {price_months[0]} : {previous_price:.6f} CHF")
print(f"Cours fin {price_months[1]} : {current_price:.6f} CHF")
print(f"Rendement = {current_price:.6f} / {previous_price:.6f} - 1 = {manual_return:.6f}")
print(f"Même valeur dans le panneau : {panel_return:.6f}")
print(f"Écart : {manual_return - panel_return:.2e}")


NOVN.SW
Cours fin 2016-09 : 64.010887 CHF
Cours fin 2016-10 : 58.983856 CHF
Rendement = 58.983856 / 64.010887 - 1 = -0.078534
Même valeur dans le panneau : -0.078534
Écart : 0.00e+00


## 3. Du SARON annuel au rendement du mois

$$
r_{f,t} = \left(1 + \frac{\mathrm{SARON}_t}{100}\right)^{1/12} - 1
$$

Le taux du fichier est un pourcentage par an. Le nombre obtenu est le rendement sans risque du mois, soustrait au rendement du portefeuille.

In [4]:
annual_percent = float(saron.loc[saron["date"].eq(first_return), "value"].iloc[0])
risk_free_month = (1 + annual_percent / 100) ** (1 / 12) - 1
published_risk_free = float(
    monthly.loc[
        monthly["month"].eq(first_return) & monthly["strategy"].eq("equal_weight"),
        "risk_free_return",
    ].iloc[0]
)
print(f"SARON {first_return} : {annual_percent:.6f} % par an")
print(f"Rendement mensuel = (1 + {annual_percent:.6f} / 100) ^ (1/12) - 1 = {risk_free_month:.8f}")
print(f"Valeur publiée : {published_risk_free:.8f}")
print(f"Écart : {risk_free_month - published_risk_free:.2e}")


SARON 2016-10 : -0.729412 % par an
Rendement mensuel = (1 + -0.729412 / 100) ^ (1/12) - 1 = -0.00060988
Valeur publiée : -0.00060988
Écart : -2.93e-18


## 4. Rendement équipondéré du même mois

Avec des poids égaux, le rendement du portefeuille est la moyenne des rendements des titres :

$$
R_{p,t} = \frac{1}{N} \sum_{i=1}^{N} R_{i,t}
$$

L'excédent retire le rendement mensuel du SARON calculé ci-dessus.

In [5]:
equal_count = len(included)
equal_weight_value = 1 / equal_count
cross_section = asset_returns.loc[first_return]
portfolio_month = float(cross_section.mean())
published_month = float(
    monthly.loc[
        monthly["month"].eq(first_return) & monthly["strategy"].eq("equal_weight"),
        "portfolio_return",
    ].iloc[0]
)
excess_month = portfolio_month - risk_free_month
print(f"N = {equal_count}")
print(f"Poids = 1 / {equal_count} = {equal_weight_value:.8f}")
print(f"Moyenne des {equal_count} rendements en {first_return} = {portfolio_month:.8f}")
print(f"Rendement publié du portefeuille = {published_month:.8f}")
print(f"Excédent = {portfolio_month:.8f} - ({risk_free_month:.8f}) = {excess_month:.8f}")


N = 83
Poids = 1 / 83 = 0.01204819
Moyenne des 83 rendements en 2016-10 = -0.02407463
Rendement publié du portefeuille = -0.02407463
Excédent = -0.02407463 - (-0.00060988) = -0.02346474


## 5. Rendement annualisé, volatilité et Sharpe

Les formules portent sur tous les mois de la fenêtre. Le rendement annualisé est géométrique. La volatilité annualisée est l'écart-type des rendements du portefeuille, multiplié par $\sqrt{12}$. Le Sharpe utilise la moyenne et l'écart-type de l'excédent, puis la même annualisation.

$$
R_p^{\mathrm{ann}} = \left(\prod_t (1+R_{p,t})\right)^{12/T} - 1
$$

$$
\sigma_p^{\mathrm{ann}} = s_R \times \sqrt{12}
\qquad
\mathrm{Sharpe}_p = \frac{\bar E_p}{s_E} \times \sqrt{12}
$$

In [6]:
def ingredients(strategy):
    series = monthly.loc[monthly["strategy"].eq(strategy)].sort_values("month")
    returns = series["portfolio_return"].to_numpy()
    excess = series["excess_return"].to_numpy()
    growth = float(np.prod(1 + returns))
    return {
        "strategy": strategy,
        "months": len(returns),
        "growth": growth,
        "annualized_return": growth ** (12 / len(returns)) - 1,
        "return_std": float(pd.Series(returns).std(ddof=1)),
        "annualized_volatility": float(pd.Series(returns).std(ddof=1) * np.sqrt(12)),
        "excess_mean": float(pd.Series(excess).mean()),
        "excess_std": float(pd.Series(excess).std(ddof=1)),
        "sharpe_ratio": float(pd.Series(excess).mean() / pd.Series(excess).std(ddof=1) * np.sqrt(12)),
    }

detail = pd.DataFrame([ingredients(strategy) for strategy in (
    "equal_weight",
    "minimum_volatility",
    "maximum_sharpe",
)])
published = metrics.set_index("strategy")
for column in ("annualized_return", "annualized_volatility", "sharpe_ratio"):
    gap = detail.set_index("strategy")[column] - published[column]
    print(f"Écart maximal sur {column} : {gap.abs().max():.2e}")
detail


Écart maximal sur annualized_return : 4.44e-16
Écart maximal sur annualized_volatility : 1.11e-16
Écart maximal sur sharpe_ratio : 6.66e-16


,strategy,months,growth,annualized_return,return_std,annualized_volatility,excess_mean,excess_std,sharpe_ratio
0,equal_weight,120,2.512330,0.096498,0.036377,0.126014,0.008459,0.036450,0.803942
1,minimum_volatility,120,1.440376,0.037164,0.021844,0.075670,0.003382,0.021932,0.534118
2,maximum_sharpe,120,17.243573,0.329422,0.046079,0.159621,0.025146,0.046108,1.889185


## 6. Variance du portefeuille

La volatilité minimale minimise $w^\top \Sigma w$. $\Sigma$ est la covariance d'échantillon des rendements des titres. La cellule calcule cette variance pour l'équipondéré et pour les poids publiés de la volatilité minimale. La racine de la variance, multipliée par $\sqrt{12}$, retrouve la volatilité annualisée.

In [7]:
covariance = asset_returns.cov()

def portfolio_variance(strategy):
    vector = (
        weights.loc[weights["strategy"].eq(strategy)]
        .set_index("yahoo_symbol")["weight"]
        .reindex(asset_returns.columns)
        .to_numpy()
    )
    return float(vector @ covariance.to_numpy() @ vector)

variance_table = pd.DataFrame(
    {
        "strategy": ["equal_weight", "minimum_volatility", "maximum_sharpe"],
        "monthly_variance": [portfolio_variance(strategy) for strategy in (
            "equal_weight",
            "minimum_volatility",
            "maximum_sharpe",
        )],
    }
)
variance_table["annualized_volatility_from_covariance"] = np.sqrt(
    variance_table["monthly_variance"]
) * np.sqrt(12)
variance_table["published_annualized_volatility"] = variance_table["strategy"].map(
    published["annualized_volatility"]
)
variance_table


,strategy,monthly_variance,annualized_volatility_from_covariance,published_annualized_volatility
0,equal_weight,0.001323,0.126014,0.126014
1,minimum_volatility,0.000477,0.075670,0.075670
2,maximum_sharpe,0.002123,0.159621,0.159621


## 7. Baisse maximale

La richesse commence à 1 avant octobre 2016 et capitalise les rendements mensuels. La baisse d'un mois est la richesse divisée par le plus haut atteint jusque-là, moins un. La baisse maximale est la plus faible de ces valeurs.

In [8]:
drawdowns = []
for strategy in ("equal_weight", "minimum_volatility", "maximum_sharpe"):
    series = (
        monthly.loc[monthly["strategy"].eq(strategy)]
        .sort_values("month")
        .set_index("month")["portfolio_return"]
    )
    wealth = (1 + series).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    worst_month = drawdown.idxmin()
    drawdowns.append(
        {
            "strategy": strategy,
            "final_wealth": float(wealth.iloc[-1]),
            "worst_month": worst_month,
            "maximum_drawdown": float(drawdown.loc[worst_month]),
            "published_maximum_drawdown": float(published.loc[strategy, "maximum_drawdown"]),
        }
    )
pd.DataFrame(drawdowns)


,strategy,final_wealth,worst_month,maximum_drawdown,published_maximum_drawdown
0,equal_weight,2.512330,2020-03,-0.167953,-0.167953
1,minimum_volatility,1.440376,2023-10,-0.165384,-0.165384
2,maximum_sharpe,17.243573,2018-12,-0.127901,-0.127901


## 8. Nombre effectif de titres

$$
N_{\mathrm{eff}} = \frac{1}{\sum_i w_i^2}
$$

L'équipondéré donne $N$. Un portefeuille concentré donne un nombre plus petit.

In [9]:
effective = []
for strategy in ("equal_weight", "minimum_volatility", "maximum_sharpe"):
    vector = weights.loc[weights["strategy"].eq(strategy), "weight"].to_numpy()
    effective.append(
        {
            "strategy": strategy,
            "sum_of_weights": float(vector.sum()),
            "maximum_weight": float(vector.max()),
            "effective_holdings": float(1 / np.square(vector).sum()),
            "names_at_or_above_1_percent": int((vector >= 0.01).sum()),
            "published_effective_holdings": float(published.loc[strategy, "effective_holdings"]),
        }
    )
pd.DataFrame(effective)


,strategy,sum_of_weights,maximum_weight,effective_holdings,names_at_or_above_1_percent,published_effective_holdings
0,equal_weight,1.0,0.012048,83.000000,83,83.000000
1,minimum_volatility,1.0,0.148874,14.089540,22,14.089540
2,maximum_sharpe,1.0,0.196590,9.572507,17,9.572507
